In [ ]:
# Lab 9, notebook 1: check the new tickets, keep the good ones, park the bad ones, and log the run.
# Attached lakehouse: freshcart_lake. Run it top to bottom; then run it again and compare.

### What arrived from the copy job

In [ ]:
import time, json
from pyspark.sql import functions as F
from delta.tables import DeltaTable

started = time.time()
raw = spark.read.table("tickets_raw")
print("rows in tickets_raw:", raw.count())
display(raw.groupBy(F.to_date("created_at").alias("day")).count().orderBy(F.desc("day")).limit(3))
# The source changed TKT-0000042's refund to 999.00 after the first copy. What does our copy say?
display(raw.filter("ticket_id = 'TKT-0000042'").select("ticket_id", "created_at", "refund_amount"))

### The rules every ticket must pass, written once

In [ ]:
# Each name is the problem a failing ticket has; each condition is what a good ticket passes.
KNOWN_CATEGORIES = ["missing_item", "late_delivery", "payment", "spillage", "other", "account", "food_safety", "wrong_item"]
RULES = {
    "no customer": "customer_id IS NOT NULL",
    "negative refund": "refund_amount >= 0",
    "unknown category": "category IN ({})".format(", ".join(f"'{c}'" for c in KNOWN_CATEGORIES)),
    "resolved before first response": "resolution_minutes >= first_response_minutes",
}
RULES_VERSION = "v1"

### Only the tickets we have not handled before

In [ ]:
handled = spark.read.table("tickets_clean").select("ticket_id") if spark.catalog.tableExists("tickets_clean") else None
if handled is not None and spark.catalog.tableExists("tickets_quarantine"):
    handled = handled.union(spark.read.table("tickets_quarantine").select("ticket_id"))
new = raw if handled is None else raw.join(handled, "ticket_id", "left_anti")
new = new.dropDuplicates(["ticket_id"])           # the same ticket twice in one batch is kept once
new_count = new.count()
print("new tickets to check:", new_count)

### Check each ticket against every rule

In [ ]:
# A rule that cannot be judged (a missing value) counts as failed.
failed = F.filter(F.array(*[F.when(~F.coalesce(F.expr(rule), F.lit(False)), F.lit(name)) for name, rule in RULES.items()]),
                  lambda x: x.isNotNull())
checked = (new.withColumn("failed_rules", failed)
              .withColumn("_checked_at", F.current_timestamp())
              .withColumn("_rules_version", F.lit(RULES_VERSION))
              .withColumn("_source", F.lit("tickets_raw, copied from freshcart_lab7.support_tickets")))
good = checked.filter(F.size("failed_rules") == 0).drop("failed_rules")
bad = checked.filter(F.size("failed_rules") > 0)
display(bad.select(F.explode("failed_rules").alias("rule")).groupBy("rule").count())
bad = bad.withColumn("failed_rules", F.concat_ws("; ", "failed_rules"))   # as text, so SQL and agents can read it

### Merge into the clean table and the quarantine, keyed on ticket_id

In [ ]:
def merge(frame, table):
    """Add the new tickets, keyed on ticket_id. The step above already removed tickets we hold, so this only
    inserts; the merge is the second guard: a ticket seen in an earlier run is never stored twice."""
    if not spark.catalog.tableExists(table):
        frame.limit(0).write.format("delta").saveAsTable(table)
    (DeltaTable.forName(spark, table).alias("t")
        .merge(frame.alias("s"), "t.ticket_id = s.ticket_id")
        .whenNotMatchedInsertAll().execute())

merge(good, "tickets_clean")
merge(bad, "tickets_quarantine")
clean_total, parked_total = spark.read.table("tickets_clean").count(), spark.read.table("tickets_quarantine").count()
print("tickets_clean:", clean_total, "| tickets_quarantine:", parked_total)

### Lineage: every change to the clean table, with what it did

In [ ]:
display(spark.sql("DESCRIBE HISTORY tickets_clean")
        .select("version", "timestamp", "operation",
                F.col("operationMetrics.numTargetRowsInserted").alias("inserted"),
                F.col("operationMetrics.numTargetRowsUpdated").alias("updated")))

### Log this run, so tomorrow you can see what happened without opening the notebook

In [ ]:
run = {"run_at": time.strftime("%Y-%m-%d %H:%M:%S"), "notebook": "lab9_1_quality", "rules_version": RULES_VERSION,
       "new_rows": new_count, "clean_total": clean_total, "quarantine_total": parked_total,
       "seconds": round(time.time() - started, 1)}
spark.createDataFrame([run]).write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable("run_log")
print(json.dumps(run))
display(spark.read.table("run_log").orderBy(F.desc("run_at")))